# CallCenterEN — Transcript Dataset Analysis

**Dataset:** [AIxBlock/92k-real-world-call-center-scripts-english](https://huggingface.co/datasets/AIxBlock/92k-real-world-call-center-scripts-english)  
**Size:** 91,706 transcripts (~10,448 audio hours)  
**Relevance to Tasknova:** Sales/support call NLP pipeline, objection handling, intent detection, talk-listen ratio  

This notebook loads the dataset, runs sanity checks, and performs basic exploratory analysis.

## 1. Setup & Installation

In [ ]:
!pip install -q datasets pandas matplotlib seaborn wordcloud

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
pd.set_option('display.max_colwidth', 120)

## 2. Load Dataset

In [ ]:
from datasets import load_dataset

# Load a streaming subset first to inspect schema without downloading full dataset
ds_stream = load_dataset(
    "AIxBlock/92k-real-world-call-center-scripts-english",
    streaming=True,
    split="train"
)

# Peek at the first record to understand schema
sample = next(iter(ds_stream))
print("=== Column names ===")
print(list(sample.keys()))
print("\n=== Sample record ===")
for k, v in sample.items():
    preview = str(v)[:200] if isinstance(v, str) else v
    print(f"  {k}: {preview}")

In [ ]:
# Load a working subset (first 10,000 records) for analysis
# For full dataset, remove the select() or increase the count
SAMPLE_SIZE = 10_000

ds = load_dataset(
    "AIxBlock/92k-real-world-call-center-scripts-english",
    split=f"train[:{SAMPLE_SIZE}]"
)

df = ds.to_pandas()
print(f"Loaded {len(df):,} records")
print(f"Columns: {list(df.columns)}")
df.head(3)

## 3. Sanity Checks

In [ ]:
print("=" * 60)
print("SANITY CHECK 1: Data types and shape")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
print("=" * 60)
print("SANITY CHECK 2: Missing values")
print("=" * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_report[missing_report.missing_count > 0] if missing.sum() > 0 else "No missing values found.")

In [ ]:
print("=" * 60)
print("SANITY CHECK 3: Duplicate records")
print("=" * 60)

# Check for exact duplicates across all columns
n_dupes = df.duplicated().sum()
print(f"Exact duplicates (all columns): {n_dupes} ({n_dupes/len(df)*100:.2f}%)")

# Check for duplicate transcript texts if a text column exists
text_cols = [c for c in df.columns if df[c].dtype == 'object']
for col in text_cols[:3]:  # check first 3 text columns
    n_text_dupes = df[col].duplicated().sum()
    print(f"Duplicate values in '{col}': {n_text_dupes} ({n_text_dupes/len(df)*100:.2f}%)")

In [ ]:
print("=" * 60)
print("SANITY CHECK 4: Empty / very short text fields")
print("=" * 60)

for col in text_cols:
    if df[col].dtype == 'object':
        lengths = df[col].astype(str).str.len()
        empty = (lengths == 0).sum()
        very_short = (lengths < 20).sum()
        print(f"Column '{col}':")
        print(f"  Empty: {empty}")
        print(f"  < 20 chars: {very_short}")
        print(f"  Median length: {lengths.median():.0f} chars")
        print(f"  Mean length: {lengths.mean():.0f} chars")
        print(f"  Max length: {lengths.max():,} chars")
        print()

In [ ]:
print("=" * 60)
print("SANITY CHECK 5: Categorical field distributions")
print("=" * 60)

for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique < 50:  # likely categorical
        print(f"\n'{col}' — {n_unique} unique values:")
        print(df[col].value_counts().head(15))

## 4. Basic Exploratory Analysis

In [ ]:
# Identify the main transcript/text column dynamically
# Common names: 'text', 'transcript', 'dialogue', 'conversation', 'content'
transcript_col = None
for candidate in ['transcript', 'text', 'dialogue', 'conversation', 'content', 'script']:
    matches = [c for c in df.columns if candidate in c.lower()]
    if matches:
        transcript_col = matches[0]
        break

if transcript_col is None:
    # Fallback: pick the longest-average text column
    avg_lens = {c: df[c].astype(str).str.len().mean() for c in text_cols}
    transcript_col = max(avg_lens, key=avg_lens.get) if avg_lens else text_cols[0]

print(f"Using '{transcript_col}' as the main transcript column")

In [ ]:
# Transcript length distribution
df['_transcript_len'] = df[transcript_col].astype(str).str.len()
df['_word_count'] = df[transcript_col].astype(str).str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['_transcript_len'], bins=80, color='steelblue', edgecolor='white')
axes[0].set_title('Transcript Length (characters)')
axes[0].set_xlabel('Characters')
axes[0].set_ylabel('Count')
axes[0].axvline(df['_transcript_len'].median(), color='red', linestyle='--', label=f"Median: {df['_transcript_len'].median():,.0f}")
axes[0].legend()

axes[1].hist(df['_word_count'], bins=80, color='darkorange', edgecolor='white')
axes[1].set_title('Transcript Length (words)')
axes[1].set_xlabel('Words')
axes[1].set_ylabel('Count')
axes[1].axvline(df['_word_count'].median(), color='red', linestyle='--', label=f"Median: {df['_word_count'].median():,.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(df[['_transcript_len', '_word_count']].describe().round(1))

In [ ]:
# Domain / category distribution (if domain column exists)
domain_col = None
for candidate in ['domain', 'category', 'topic', 'type', 'call_type', 'service']:
    matches = [c for c in df.columns if candidate in c.lower()]
    if matches:
        domain_col = matches[0]
        break

if domain_col:
    fig, ax = plt.subplots(figsize=(10, 6))
    counts = df[domain_col].value_counts().head(15)
    counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'Distribution by {domain_col}')
    ax.set_xlabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(v + len(df)*0.005, i, f"{v:,} ({v/len(df)*100:.1f}%)", va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("No domain/category column found — skipping domain distribution chart.")

In [ ]:
# Inbound vs Outbound split (if direction column exists)
direction_col = None
for candidate in ['direction', 'inbound', 'outbound', 'call_direction']:
    matches = [c for c in df.columns if candidate in c.lower()]
    if matches:
        direction_col = matches[0]
        break

if direction_col:
    fig, ax = plt.subplots(figsize=(6, 4))
    df[direction_col].value_counts().plot(kind='pie', autopct='%1.1f%%', ax=ax,
                                          colors=['#4C72B0', '#DD8452'])
    ax.set_title('Inbound vs Outbound')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("No direction column found — skipping inbound/outbound chart.")

In [ ]:
# Turn-level analysis: count agent vs customer turns
# Common patterns: "Agent:", "Customer:", "Speaker 1:", etc.
sample_text = df[transcript_col].astype(str).iloc[0]
print("First 500 chars of sample transcript (to identify turn markers):")
print(sample_text[:500])
print("...")

In [ ]:
# Attempt to extract speaker turns from transcript text
# Adapt regex based on observed format above
def count_turns(text):
    text = str(text)
    # Try common patterns
    agent_patterns = [r'\bAgent:', r'\bRep:', r'\bSalesperson:', r'\bAdvisor:', r'\bSpeaker ?1:']
    customer_patterns = [r'\bCustomer:', r'\bClient:', r'\bCaller:', r'\bSpeaker ?2:']

    agent_count = sum(len(re.findall(p, text, re.IGNORECASE)) for p in agent_patterns)
    customer_count = sum(len(re.findall(p, text, re.IGNORECASE)) for p in customer_patterns)
    return agent_count, customer_count

turns = df[transcript_col].apply(count_turns)
df['_agent_turns'] = turns.apply(lambda x: x[0])
df['_customer_turns'] = turns.apply(lambda x: x[1])
df['_total_turns'] = df['_agent_turns'] + df['_customer_turns']

print("Turn counts (agent vs customer):")
print(df[['_agent_turns', '_customer_turns', '_total_turns']].describe().round(1))

if df['_total_turns'].sum() > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df['_total_turns'][df['_total_turns'] > 0], bins=50, color='steelblue', edgecolor='white')
    ax.set_title('Total Turns per Conversation')
    ax.set_xlabel('Number of turns')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo speaker turn markers detected — transcript may use a different format.")
    print("Check the sample above and adjust the regex patterns.")

In [ ]:
# Word frequency analysis (top terms across all transcripts)
from collections import Counter

STOP_WORDS = {'the', 'a', 'an', 'is', 'it', 'to', 'and', 'of', 'in', 'for', 'on',
              'that', 'this', 'with', 'you', 'i', 'we', 'are', 'was', 'be', 'have',
              'has', 'had', 'do', 'does', 'did', 'will', 'would', 'can', 'could',
              'not', 'but', 'or', 'if', 'so', 'just', 'my', 'your', 'me', 'our',
              'am', 'been', 'being', 'no', 'yes', 'okay', 'ok', 'um', 'uh', 'like',
              'right', 'well', 'yeah', 'gonna', 'wanna', 'got', 'get', 'go', 'know',
              'at', 'from', 'as', 'up', 'out', 'about', 'what', 'there', 'them',
              'they', 'he', 'she', 'his', 'her', 'its', 'all', 'also'}

all_words = Counter()
for text in df[transcript_col].astype(str).sample(min(5000, len(df))):
    words = re.findall(r'\b[a-z]{3,}\b', text.lower())
    all_words.update(w for w in words if w not in STOP_WORDS)

top_30 = all_words.most_common(30)
words_df = pd.DataFrame(top_30, columns=['word', 'count'])

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(words_df['word'][::-1], words_df['count'][::-1], color='teal')
ax.set_title('Top 30 Words (excluding stop words)')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# PII redaction check — look for potential leaked PII patterns
print("=" * 60)
print("SANITY CHECK 6: PII leakage scan")
print("=" * 60)

pii_patterns = {
    'phone_numbers': r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
    'email_addresses': r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
    'ssn_patterns': r'\b\d{3}-\d{2}-\d{4}\b',
    'credit_card_like': r'\b\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}\b',
}

sample_texts = df[transcript_col].astype(str).sample(min(5000, len(df)))
combined = ' '.join(sample_texts)

for name, pattern in pii_patterns.items():
    matches = re.findall(pattern, combined)
    status = f"FOUND {len(matches)} matches" if matches else "Clean"
    print(f"  {name}: {status}")

# Check for redaction markers
redaction_markers = re.findall(r'\[REDACTED\]|\[PII\]|\*\*\*|<PII>|\[REMOVED\]', combined, re.IGNORECASE)
print(f"\n  Redaction markers found: {len(redaction_markers)}")
print(f"  (Indicates PII scrubbing was applied)" if redaction_markers else "  (No explicit markers — PII may have been replaced or removed silently)")

## 5. Summary

**Findings from this analysis:**

| Check | Status |
|---|---|
| Schema loaded | See columns above |
| Missing values | See check 2 |
| Duplicates | See check 3 |
| Empty transcripts | See check 4 |
| Domain balance | See chart |
| PII leakage | See check 6 |

**Relevance to Tasknova:** This dataset provides the largest available corpus of real customer service transcripts with Indian accents. It can be used to train the NLP pipeline for intent detection, objection handling classification, and sentiment scoring. The lack of audio limits its use for ASR training — pair with IndicVoices for that.